# Reservoirs in Catalunya
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 15-03-2024<br>

**Introduction:**<br>
This code preprocesses the reservoir time series downloaded from [Agència Catalana de l'Aigua](https://analisi.transparenciacatalunya.cat/es/Medi-Ambient/Xarxes-de-control-del-medi-consulta-de-l-aigua-i-e/wc95-u57z/about_data). The raw data includes reservoir attributes (coordinates, ID, name) and daily time series of reservoir storage, level and fraction filled. The results of the code are a CSV file with the reservoir attributes, and several CSV files (one for each reservoir) with the daily timeseries.

> **Note**. Later I've added manually data to the attributes table extracted from the CEDEX dataset.

In [1]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# from ocab.config import Config
from ocab.anuario.stations.reservoirs import get_reservoirs_aca

## Configuration

In [2]:
# # load configuration of the BEAVERS dataset
# cfg = Config('../../BEAVERS/config_BEAVERS_v100.yml')

# path where the data is stored
path_aca = Path('/home/casadoj/Data/ACA')

# paths where results will be saved
path_results = path_aca / 'processed' / 'reservoirs'
path_gis = path_results / 'GIS'
path_ts = path_results / 'timeseries'
for path in [path_gis, path_ts]:
    path.mkdir(parents=True, exist_ok=True)

## ACA raw data

Here I load and prepocess the raw Excel files downloaded from the ACA website. The result are two objects: `timeseries` is a dictionary that contains the time series for each reservoir, `reservoirs_aca` is a DataFrame with the reservoir attributes.

### Time series

In [5]:
# time series will be saved in a dictionary and reservoir attributes in a pandas.DataFrame
timeseries = {}
dams = pd.DataFrame()

# read raw Excel files iteratively
path_in = path_aca / 'raw' / 'reservoirs'
files = sorted(list(path_in.glob('reservoirs_*.xlsx')))
for file in tqdm(files, desc='files'):
    
    # # check data is inside the dataset period
    # start, end = [int(year) for year in file.stem.split('_')[1:]]
    # if (start > cfg.end.year) or (end < cfg.start.year):
    #     continue

    # get reservoir attributes and time series
    attrs, ts = get_reservoirs_aca(file)

    # update reservoirs
    dams = pd.concat([dams, attrs], axis=0).drop_duplicates()

    # update time series
    for ID in ts:
        if ID in timeseries:
            timeseries[ID] = pd.concat((timeseries[ID], ts[ID]), axis=0).sort_index(axis=0)
        else:
            timeseries[ID] = ts[ID]

dams.index.name = 'id_aca'

files:   0%|          | 0/8 [00:00<?, ?it/s]

## Export

### Attributes

In [6]:
dams.to_file(path_gis / 'dams_aca.geojson', driver='GeoJSON')

### Time series

In [7]:
for ID, ts in timeseries.items():
    ts.to_parquet(path_ts / f'{ID}.parquet')